In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import re


import pandas as pd

# Pedir el nombre del archivo .xlsx
#archivo = input("Introduce el nombre del archivo .xlsx (incluyendo la extensión): ")

# Leer el archivo
df = pd.read_excel('/content/drive/MyDrive/Desarrollo/Pruebas/nuevopruebas/df_dropclas.xlsx')


# Filtrar las filas donde el texto en la columna especificada comienza con "Nombre:"
# Reemplaza 'Descripcion' con el nombre de la columna donde están los datos a filtrar
columna_datos = 'Descripción de la orden'
df_filtrado = df[df[columna_datos].astype(str).str.startswith("Nombre:")]

#print(df_filtrado)


# Función para extraer los campos de cada texto
def extract_fields(text):
    nombre = re.search(r'Nombre:\s*([\w\s]+),', text)
    telefono = re.search(r'Teléfono:\s*(\d+),', text)
    direccion = re.search(r'Dirección Predio:\s*([\w\s#-]+),', text)
    municipio = re.search(r'Municipio:\s*([\w\s]+),', text)
    observacion = re.search(r'Observación Detalle de ubicación:\s*([\w\s]+),', text)
    tipo_dano = re.search(r'Tipo de daño:\s*([\w\s]+),', text)
    detalle_dano = re.search(r'Detalle de daño:\s*([\w\s]+)', text)

    # Extraer valores de cada campo, asignando None si no se encuentra el valor
    nombre = nombre.group(1) if nombre else None
    telefono = telefono.group(1) if telefono else None
    direccion = direccion.group(1) if direccion else None
    municipio = municipio.group(1) if municipio else None
    observacion = observacion.group(1) if observacion else None
    tipo_dano = tipo_dano.group(1) if tipo_dano else None
    detalle_dano = detalle_dano.group(1) if detalle_dano else None

    return pd.Series([nombre, telefono, direccion, municipio, observacion, tipo_dano, detalle_dano])

# Aplicar la función a cada fila de la columna 'descripcion' y crear un nuevo DataFrame
df_extraccion = df_filtrado['Descripción de la orden'].apply(extract_fields)
df_extraccion.columns = ["Nombre", "Teléfono", "Dirección Predio", "Municipio", "Observación", "Tipo de daño", "Detalle de daño"]



df1 =  pd.concat([df_filtrado, df_extraccion], axis=1)

print(df1)
# Mostrar el DataFrame resultante
#print(df_extraccion)
columnas = ["Nombre", "Teléfono", "Dirección Predio", "Municipio", "Observación", "Tipo de daño", "Detalle de daño", "out"]

dft = df1[columnas]

columna_datos = 'out'
dft = dft[~dft[columna_datos].str.contains('pendiente', case=False, na=False)]

dft.info()


In [ ]:
dft.head(100)

In [ ]:
pip install datasets

In [ ]:
dft['out'] = dft['out'].str.lower()
dft['Detalle de daño'] = dft['Detalle de daño'].str.lower()

dft = dft.dropna(subset=['Detalle de daño'])

cols = ["Detalle de daño","out"]

dfn = dft[cols]

dfn.head(100)

In [ ]:
conteo_clases = dfn["out"].value_counts()
print(conteo_clases)

In [ ]:
#max_count = dfn["out"].value_counts().max()
# Definir el número máximo de muestras por clase
max_count = 5000
# Sobremuestrear cada clase para tener la misma cantidad de datos que la clase mayoritaria
dfb = dfn.groupby("out").apply(lambda x: x.sample(n=max_count, replace=True, random_state=42)).reset_index(drop=True)
print(dfb["out"].value_counts())

In [ ]:
dfb.info()

In [ ]:
pip install --upgrade wandb


In [ ]:
from collections import Counter
from sklearn.model_selection import train_test_split
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, TrainingArguments, Trainer
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score
import pandas as pd
import wandb
from transformers.integrations import WandbCallback

# Inicializar WandB
wandb.init(
    project="Caller-Classification",  # Nombre del proyecto en WandB
    name="Entrenamiento DistilBERT",  # Título del experimento
    notes="Modelo entrenado por [Camilo, Cesar, Diego, Rober]",  # Notas sobre el grupo
    config={
        "learning_rate": 2e-5,
        "batch_size": 8,
        "epochs": 7,
        "model": "distilbert-base-uncased"
    }
)

# Cargar los datos
# dfb = pd.read_csv("tu_archivo.csv")  # Descomenta y ajusta si necesitas cargar desde un archivo

# Seleccionar la columna de texto y la columna de etiquetas
texts = dfb["Detalle de daño"].tolist()   # Columna de fallas
labels = dfb["out"].tolist()              # Columna de clases

# Convertir etiquetas a índices numéricos (necesario si son texto)
label_mapping = {"ejecutar": 0, "cancelar": 1}
labels = [label_mapping[label] for label in labels]

# Evaluar balance de clases
class_counts = Counter(labels)
print(f"Balance de clases: {class_counts}")

# Dividir el dataset en entrenamiento y validación
train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)

# Procesar los textos (limpieza, stopwords, lematización)
def procesar_texto(texto):
    # Implementa tu función de preprocesamiento aquí si es necesario
    return texto

train_texts = [procesar_texto(texto) for texto in train_texts]
val_texts = [procesar_texto(texto) for texto in val_texts]

# Inicializar el tokenizer y el modelo de DistilBERT
tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=2
)

# Ajustar el dropout manualmente
model.config.hidden_dropout_prob = 0.2

# Tokenizar los datos de entrenamiento y validación
train_encodings = tokenizer(train_texts, padding=True, truncation=True, max_length=128)
val_encodings = tokenizer(val_texts, padding=True, truncation=True, max_length=128)

# Crear datasets de Hugging Face
train_dataset = Dataset.from_dict({
    "input_ids": train_encodings["input_ids"],
    "attention_mask": train_encodings["attention_mask"],
    "labels": train_labels,
})
val_dataset = Dataset.from_dict({
    "input_ids": val_encodings["input_ids"],
    "attention_mask": val_encodings["attention_mask"],
    "labels": val_labels,
})

# Crear un DatasetDict para manejo estructurado
datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

# Configurar los argumentos de entrenamiento
training_args = TrainingArguments(
    output_dir="./modelo1",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    num_train_epochs=7,
    weight_decay=0.05,
    evaluation_strategy="epoch",
    logging_dir="./logs",
    logging_steps=10,
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to=["wandb"],
)

# Definir métricas
def compute_metrics(p):
    preds = p.predictions.argmax(axis=-1)
    labels = p.label_ids
    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds)
    wandb.log({"accuracy": acc, "f1": f1})
    return {"accuracy": acc, "f1": f1}

# Inicializar el entrenador
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

# Entrenar el modelo
trainer.train()

# Evaluar el modelo en el conjunto de validación
trainer.evaluate()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast, TrainingArguments, Trainer
# ------------------------------------------------------------------------------------------------------
# Cargar el modelo y el tokenizador desde el directorio guardado
model = DistilBertForSequenceClassification.from_pretrained("/content/drive/MyDrive/Desarrollo/Pruebas/modelo1")
tokenizer = DistilBertTokenizerFast.from_pretrained("/content/drive/MyDrive/Desarrollo/Pruebas/modelo1")



In [ ]:
#Importar librerías
from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
import pandas as pd
import torch
import seaborn as sns
import matplotlib.pyplot as plt

Cargar Dataset

In [ ]:
df_test = pd.read_csv('/content/drive/MyDrive/Desarrollo/Pruebas/pruba19/df.csv')

Preprocesamiento del Nuevo Dataset

In [ ]:
df_test['out'].value_counts().to_frame().sort_index()

In [ ]:
# Calcular proporciones originales
class_counts = df_test["out"].value_counts()
total = class_counts.sum()
ratios = class_counts / total

# Calcular número mínimo de muestras basado en la clase menos frecuente
min_samples = min(class_counts)

# Crear un nuevo DataFrame ajustado para balancear según la clase menos frecuente
dfb = pd.concat([
    df_test[df_test["out"] == label].sample(
        n=min_samples,
        replace=False,  # No repetir muestras
        random_state=42
    )
    for label in class_counts.index
]).reset_index(drop=True)

# Verificar el balanceo final
print(dfb["out"].value_counts())


In [ ]:
# Extraer los textos y las etiquetas
test_texts = dfb['Detalle de daño'].tolist()
test_labels = dfb['out'].tolist()

# Mapear las etiquetas a índices numéricos
label_mapping = {"ejecutar": 0, "cancelar": 1}  # Ajusta este mapeo si es necesario
test_labels = [label_mapping[label] for label in test_labels]


In [ ]:
# Eliminar cadenas vacías o None
test_texts = [texto for texto in test_texts if texto and isinstance(texto, str)]

Tokenizar el Nuevo Dataset

In [ ]:
!pip install datasets

In [ ]:
filtered_test_texts = [texto for texto in test_texts if texto and isinstance(texto, str)]
filtered_test_labels = [label for texto, label in zip(test_texts, test_labels) if texto and isinstance(texto, str)]

In [ ]:
assert len(filtered_test_texts) == len(filtered_test_labels), "Las longitudes de los textos y las etiquetas no coinciden"

In [ ]:
# Tokenizar el conjunto de prueba
test_encodings = tokenizer(filtered_test_texts, padding=True, truncation=True, max_length=128, return_tensors="pt")

# Crear el dataset de Hugging Face para el conjunto de prueba
from datasets import Dataset

test_dataset = Dataset.from_dict({
    "input_ids": test_encodings["input_ids"].tolist(),  # Convertimos a listas para ser compatibles con Hugging Face
    "attention_mask": test_encodings["attention_mask"].tolist(),
    "labels": filtered_test_labels,  # Usamos las etiquetas filtradas
})

# Mostrar una parte del dataset para verificar que todo está correcto
print(test_dataset)


Crear el DataLoader para el Conjunto de Prueba

In [ ]:
# Crear un DataLoader para el conjunto de prueba
test_dataloader = DataLoader(test_dataset, batch_size=8)


Realizar las Predicciones en el Conjunto de Prueba

In [ ]:
from torch.utils.data import DataLoader
from transformers import DistilBertTokenizerFast
import torch

# Crear el DataLoader para el conjunto de prueba
batch_size = 8  # Ajusta el tamaño del lote según sea necesario

# Convertir el dataset de prueba en un DataLoader
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
for batch in test_dataloader:
    print(type(batch))  # Verificar si es una lista o un diccionario
    print(batch)  # Ver los primeros elementos del batch
    break  # Solo para imprimir una vez, ya que el dataloader es iterativo



In [ ]:
import torch
from torch.utils.data import DataLoader

def collate_fn(batch):
    # Convertir cada campo en un tensor (si aún no es un tensor)
    input_ids = torch.stack([torch.tensor(item['input_ids']) for item in batch])
    attention_mask = torch.stack([torch.tensor(item['attention_mask']) for item in batch])
    labels = torch.stack([torch.tensor(item['labels']) for item in batch])

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels
    }

# Asegúrate de que el dataloader utilice este 'collate_fn'
test_dataloader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)



In [ ]:
import torch

# Asegúrate de que el modelo esté en el dispositivo correcto
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)  # Mover el modelo al dispositivo adecuado

# Función para predecir con el modelo
def predict(model, dataloader):
    model.eval()  # Ponemos el modelo en modo de evaluación
    all_preds = []
    all_labels = []

    with torch.no_grad():  # Desactivamos el cálculo de gradientes para la inferencia
        for batch in dataloader:
            # Mover los tensores al dispositivo correcto (GPU o CPU)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            # Verificar en qué dispositivo están los tensores
            print(f"Modelo está en: {model.device}")
            print(f"input_ids está en: {input_ids.device}")
            print(f"attention_mask está en: {attention_mask.device}")
            print(f"labels está en: {labels.device}")

            # Realizar las predicciones
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits  # La salida del modelo
            preds = torch.argmax(logits, dim=-1)  # Predicción: la clase con la mayor probabilidad

            # Guardamos las predicciones y las etiquetas reales
            all_preds.extend(preds.cpu().numpy())  # Convertimos las predicciones a numpy
            all_labels.extend(labels.cpu().numpy())  # Convertimos las etiquetas a numpy

    return all_preds, all_labels

# Realizar las predicciones en el conjunto de prueba
predictions, true_labels = predict(model, test_dataloader)



Generar la Matriz de Confusión

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt


# Crear la matriz de confusión
cm = confusion_matrix(true_labels, predictions)

# Mostrar la matriz de confusión con una visualización más amigable
import seaborn as sns
import matplotlib.pyplot as plt

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=["ejecutar", "cancelar"], yticklabels=["ejecutar", "cancelar"])
plt.xlabel('Predicción')
plt.ylabel('Verdadero')
plt.title('Matriz de Confusión del Modelo en el Conjunto de Prueba')
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report


# Calcular métricas
accuracy = accuracy_score(true_labels, predictions)
precision = precision_score(true_labels, predictions, average='weighted')
recall = recall_score(true_labels, predictions, average='weighted')
f1 = f1_score(true_labels, predictions, average='weighted')

# Mostrar métricas
print("Métricas del Modelo:")
print(f"Precisión (Accuracy): {accuracy:.2f}")
print(f"Precisión (Precision): {precision:.2f}")
print(f"Sensibilidad (Recall): {recall:.2f}")
print(f"F1-Score: {f1:.2f}")

# Reporte detallado por clase
print("\nReporte de Clasificación:")
print(classification_report(true_labels, predictions, target_names=["ejecutar", "cancelar"]))